In [0]:
# ============================================
# FIND WHERE BRONZE TABLE IS STORED
# ============================================

print("CURRENT CATALOG:")
print(spark.catalog.currentCatalog())

print("\nCURRENT SCHEMA:")
print(spark.catalog.currentDatabase())

print("\nAVAILABLE CATALOGS:")
display(
    spark.sql("SHOW CATALOGS")
)

print("\nCURRENT SCHEMA TABLES:")
display(
    spark.sql("SHOW TABLES")
)

CURRENT CATALOG:
workspace

CURRENT SCHEMA:
default

AVAILABLE CATALOGS:


catalog
bank_data
my_data
samples
system
workspace



CURRENT SCHEMA TABLES:


database,tableName,isTemporary
default,gold_account_summary,false
default,gold_daily_banking_kpis,false
default,gold_transaction_analysis,false
default,sales_delta,false
default,sales_fact,false
default,sales_fact_training,false
default,sales_fact_v2,false
default,sales_summary,false
default,silver_bank_transactions,false


In [0]:
# ============================================
# FIND BRONZE TABLE
# ============================================

print("Schemas inside workspace:")

display(
    spark.sql("SHOW SCHEMAS IN workspace")
)

Schemas inside workspace:


databaseName
default
information_schema


In [0]:
%pip install openpyxl

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ============================================================
# BANKING DATA ENGINEERING PROJECT
# SILVER LAYER
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    to_timestamp,
    trim
)


# ============================================================
# SPARK SESSION
# ============================================================

spark = SparkSession.builder.appName(
    "Banking_Silver"
).getOrCreate()


# ============================================================
# CONFIGURATION
# ============================================================

bronze_path = (
    "/Volumes/bank_data/default/bank_data/"
    "bronze_bank_transactions"
)

silver_path = (
    "/Volumes/bank_data/default/bank_data/"
    "silver_bank_transactions"
)


# ============================================================
# READ BRONZE
# ============================================================

print("========================================")
print("         READ BRONZE")
print("========================================")

bronze_df = (
    spark.read
    .format("delta")
    .load(bronze_path)
)

bronze_count = bronze_df.count()

print(f"Bronze Rows: {bronze_count}")


# ============================================================
# SILVER TRANSFORMATION
# ============================================================

print("\n========================================")
print("      SILVER TRANSFORMATION")
print("========================================")


silver_df = (
    bronze_df

    # Remove exact duplicate records
    .dropDuplicates()

    # Remove unnecessary raw column
    .drop(".")

    # Convert DATE from string to timestamp
    .withColumn(
        "DATE",
        to_timestamp(
            col("DATE"),
            "yyyy-MM-dd HH:mm:ss"
        )
    )

    # Convert VALUE_DATE from string to timestamp
    .withColumn(
        "VALUE_DATE",
        to_timestamp(
            col("VALUE_DATE"),
            "yyyy-MM-dd HH:mm:ss"
        )
    )

    # Clean Account Number
    .withColumn(
        "Account_No",
        trim(col("Account_No"))
    )

    # Clean Transaction Details
    .withColumn(
        "TRANSACTION_DETAILS",
        trim(col("TRANSACTION_DETAILS"))
    )
)


# ============================================================
# WRITE SILVER
# ============================================================

print("\n========================================")
print("         WRITE SILVER")
print("========================================")


(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)


print(" Silver Delta data written successfully")


# ============================================================
# READ SILVER FOR VALIDATION
# ============================================================

silver_check_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)


silver_count = silver_check_df.count()

silver_columns = len(
    silver_check_df.columns
)


# ============================================================
# DUPLICATE VALIDATION
# ============================================================

print("\n========================================")
print("      DUPLICATE VALIDATION")
print("========================================")


distinct_count = silver_check_df.distinct().count()

duplicate_count = (
    silver_count - distinct_count
)


print(f"Total Silver Rows: {silver_count}")
print(f"Distinct Silver Rows: {distinct_count}")
print(f"Duplicate Rows: {duplicate_count}")


if duplicate_count == 0:

    print(
        " DUPLICATE VALIDATION PASSED"
    )

else:

    print(
        " DUPLICATE VALIDATION FAILED"
    )


# ============================================================
# SILVER VALIDATION
# ============================================================

print("\n========================================")
print("         SILVER VALIDATION")
print("========================================")


print(
    f"Bronze Rows: {bronze_count}"
)

print(
    f"Silver Rows: {silver_count}"
)

print(
    f"Silver Columns: {silver_columns}"
)


expected_silver_rows = 116162

expected_silver_columns = 10


if silver_count == expected_silver_rows:

    print(
        " SILVER ROW COUNT VALIDATION PASSED"
    )

else:

    print(
        " SILVER ROW COUNT VALIDATION FAILED"
    )


if silver_columns == expected_silver_columns:

    print(
        " SILVER COLUMN COUNT VALIDATION PASSED"
    )

else:

    print(
        " SILVER COLUMN COUNT VALIDATION FAILED"
    )


# ============================================================
# NULL VALIDATION
# ============================================================

print("\n========================================")
print("         NULL VALIDATION")
print("========================================")


null_account = (
    silver_check_df
    .filter(
        col("Account_No").isNull()
    )
    .count()
)


null_date = (
    silver_check_df
    .filter(
        col("DATE").isNull()
    )
    .count()
)


null_value_date = (
    silver_check_df
    .filter(
        col("VALUE_DATE").isNull()
    )
    .count()
)


print(
    f"NULL Account_No: {null_account}"
)

print(
    f"NULL DATE: {null_date}"
)

print(
    f"NULL VALUE_DATE: {null_value_date}"
)


if null_account == 0:

    print(
        "Account_No NULL CHECK PASSED"
    )

else:

    print(
        " Account_No NULL CHECK FAILED"
    )


if null_date == 0:

    print(
        " DATE NULL CHECK PASSED"
    )

else:

    print(
        " DATE NULL CHECK FAILED"
    )


if null_value_date == 0:

    print(
        "VALUE_DATE NULL CHECK PASSED"
    )

else:

    print(
        " VALUE_DATE NULL CHECK FAILED"
    )


# ============================================================
# CHECK DATE TYPES
# ============================================================

print("\n========================================")
print("         DATA TYPES VALIDATION")
print("========================================")


date_type = dict(
    silver_check_df.dtypes
).get("DATE")


value_date_type = dict(
    silver_check_df.dtypes
).get("VALUE_DATE")


print(
    f"DATE type: {date_type}"
)

print(
    f"VALUE_DATE type: {value_date_type}"
)


if date_type == "timestamp":

    print(
        " DATE TYPE VALIDATION PASSED"
    )

else:

    print(
        " DATE TYPE VALIDATION FAILED"
    )


if value_date_type == "timestamp":

    print(
        "VALUE_DATE TYPE VALIDATION PASSED"
    )

else:

    print(
        "VALUE_DATE TYPE VALIDATION FAILED"
    )


# ============================================================
# SILVER SCHEMA
# ============================================================

print("\n========================================")
print("         SILVER SCHEMA")
print("========================================")


silver_check_df.printSchema()


# ============================================================
# SILVER SAMPLE DATA
# ============================================================

print("\n========================================")
print("         SILVER SAMPLE DATA")
print("========================================")


display(
    silver_check_df.limit(10)
)


# ============================================================
# FINAL DATA QUALITY CHECK
# ============================================================

if (
    silver_count == expected_silver_rows
    and silver_columns == expected_silver_columns
    and duplicate_count == 0
    and null_account == 0
    and null_date == 0
    and null_value_date == 0
    and date_type == "timestamp"
    and value_date_type == "timestamp"
):

    print("\n========================================")
    print("       SILVER LAYER COMPLETED")
    print("========================================")

    print(
        " ALL SILVER DATA QUALITY CHECKS PASSED"
    )

    print(
        f"Silver Rows: {silver_count}"
    )

    print(
        f"Silver Columns: {silver_columns}"
    )

    print(
        f"Silver Delta Path: {silver_path}"
    )

else:

    raise Exception(
        " SILVER DATA QUALITY CHECK FAILED"
    )

         READ BRONZE
Bronze Rows: 116201

      SILVER TRANSFORMATION

         WRITE SILVER
 Silver Delta data written successfully

      DUPLICATE VALIDATION
Total Silver Rows: 116162
Distinct Silver Rows: 116162
Duplicate Rows: 0
 DUPLICATE VALIDATION PASSED

         SILVER VALIDATION
Bronze Rows: 116201
Silver Rows: 116162
Silver Columns: 10
 SILVER ROW COUNT VALIDATION PASSED
 SILVER COLUMN COUNT VALIDATION PASSED

         NULL VALIDATION
NULL Account_No: 0
NULL DATE: 0
NULL VALUE_DATE: 0
Account_No NULL CHECK PASSED
 DATE NULL CHECK PASSED
VALUE_DATE NULL CHECK PASSED

         DATA TYPES VALIDATION
DATE type: timestamp
VALUE_DATE type: timestamp
 DATE TYPE VALIDATION PASSED
VALUE_DATE TYPE VALIDATION PASSED

         SILVER SCHEMA
root
 |-- Account_No: string (nullable = true)
 |-- DATE: timestamp (nullable = true)
 |-- TRANSACTION_DETAILS: string (nullable = true)
 |-- CHQ.NO.: double (nullable = true)
 |-- VALUE_DATE: timestamp (nullable = true)
 |-- WITHDRAWAL_AMT: double 

Account_No,DATE,TRANSACTION_DETAILS,CHQ.NO.,VALUE_DATE,WITHDRAWAL_AMT,DEPOSIT_AMT,BALANCE_AMT,_source_file,_ingestion_timestamp
409000438620',2017-01-30T00:00:00.000Z,AEPS Indiaforensic INCOME 280117,null,2017-01-30T00:00:00.000Z,null,36755.22,-5.457223773899996E8,bank.xlsx,2026-10-06T21:12:17.712Z
409000438620',2017-01-30T00:00:00.000Z,Indiaforensic AEPS NPCI WDL SET,null,2017-01-30T00:00:00.000Z,null,3326417.0,-5.420119315699996E8,bank.xlsx,2026-10-06T21:12:17.712Z
409000438620',2017-02-03T00:00:00.000Z,Indiaforensic AEPS NPCI DEP SET,null,2017-02-03T00:00:00.000Z,1766575.0,null,-5.421870964699997E8,bank.xlsx,2026-10-06T21:12:17.712Z
409000438620',2017-02-06T00:00:00.000Z,AEPS Indiaforensic INCOME 050217,null,2017-02-06T00:00:00.000Z,null,37707.61,-5.455330387799996E8,bank.xlsx,2026-10-06T21:12:17.712Z
409000438620',2017-02-06T00:00:00.000Z,Indiaforensic AEPS NPCI DEP SET,null,2017-02-06T00:00:00.000Z,1674416.0,null,-5.397475396399996E8,bank.xlsx,2026-10-06T21:12:17.712Z
409000438620',2017-02-08T00:00:00.000Z,AEPS Indiaforensic INCOME 070217,null,2017-02-08T00:00:00.000Z,null,41601.32,-5.462287934799995E8,bank.xlsx,2026-10-06T21:12:17.712Z
409000438620',2017-02-08T00:00:00.000Z,AEPS Indiaforensic S TAX 070217,null,2017-02-08T00:00:00.000Z,null,6240.2,-5.462225532799995E8,bank.xlsx,2026-10-06T21:12:17.712Z
409000438620',2017-02-15T00:00:00.000Z,Indiaforensic AEPS NPCI DEP SET,null,2017-02-15T00:00:00.000Z,2013959.0,null,-5.411910575599998E8,bank.xlsx,2026-10-06T21:12:17.712Z
409000438620',2017-02-18T00:00:00.000Z,TRF TO Indiaforensic SERVICES I,null,2017-02-18T00:00:00.000Z,4000000.0,null,-5.420358835699999E8,bank.xlsx,2026-10-06T21:12:17.712Z
409000438620',2017-02-21T00:00:00.000Z,Indiaforensic AEPS NPCI WDL SET,null,2017-02-21T00:00:00.000Z,null,5174625.0,-5.410028893399999E8,bank.xlsx,2026-10-06T21:12:17.712Z



       SILVER LAYER COMPLETED
 ALL SILVER DATA QUALITY CHECKS PASSED
Silver Rows: 116162
Silver Columns: 10
Silver Delta Path: /Volumes/bank_data/default/bank_data/silver_bank_transactions
